# EDA A3 – Na czym zarabiamy?

Pytanie biznesowe: **które produkty i kategorie przynoszą przychód i zysk?**

Dodatkowo sprawdzamy hipotezę z A2: mężczyźni mają o 13% wyższe revenue per user wyłącznie dzięki
wyższemu AOV. **Czy to efekt zachowania klientów, czy po prostu droższego asortymentu?**

Zasady z A1:
* przychód = suma `sale_price` dla zamówień **Complete + Shipped**,
* status bierzemy z `orders` (pozycje łączymy z zamówieniem po `order_id`),
* zysk brutto = `sale_price - cost` (koszt zakupu produktu, bez logistyki i reklamy).

## Wczytanie danych

* `products` – katalog: kategoria, dział (Men / Women), marka, koszt i cena katalogowa.
* `orders` – status i płeć klienta (`orders.gender`).
* `order_items` – który produkt i za ile sprzedano w każdym zamówieniu.

In [1]:
import pandas as pd
import numpy as np

from bq_utils import read_bq

REVENUE_STATUSES = ["Complete", "Shipped"]

In [2]:
# Products

products_cols = ["id", "brand", "category", "department", "cost", "retail_price"]

products_dtypes = {"id": "Int32", "brand": "category", "category": "category",
                   "department": "category"}

products = read_bq("products", usecols=products_cols, dtype=products_dtypes)

products.head()

Zapytanie przetworzy: 1.59 MB


,id,brand,category,department,cost,retail_price
0,13842,MG,Accessories,Women,2.51875,6.25
1,13928,MG,Accessories,Women,2.33835,5.95
2,14115,MG,Accessories,Women,4.87956,10.99
3,14157,MG,Accessories,Women,4.64877,10.99
4,14273,MG,Accessories,Women,6.50793,15.99


In [3]:
# Orders – status decides whether an item counts as revenue

orders = read_bq("orders", usecols=["order_id", "status", "gender"],
                 dtype={"order_id": "Int32", "status": "category", "gender": "category"})

orders.head()

Zapytanie przetworzy: 2.64 MB


,order_id,status,gender
0,6,Cancelled,F
1,8,Cancelled,F
2,38,Cancelled,F
3,52,Cancelled,F
4,55,Cancelled,F


In [4]:
# Order items

order_items = read_bq("order_items", usecols=["order_id", "product_id", "sale_price"],
                      dtype={"order_id": "Int32", "product_id": "Int32"})

order_items.head()

Zapytanie przetworzy: 4.34 MB


,order_id,product_id,sale_price
0,2526,14235,0.02
1,18153,14235,0.02
2,47981,14235,0.02
3,84256,14235,0.02
4,348,14235,0.02


In [5]:
# Data as of

checked_at = pd.Timestamp.now(tz="UTC")
print(f"Sprawdzono: {checked_at:%Y-%m-%d %H:%M} UTC")

Sprawdzono: 2026-10-29 08:39 UTC


## 1. Asortyment

* Ile produktów jest w każdym dziale i kategorii? Jaka jest średnia cena?
* Ile jest marek i które mają najwięcej produktów?

In [6]:
# Products and average price by department

products.groupby("department", observed=True).agg(
    products=("id", "count"),
    avg_price=("retail_price", "mean"),
    median_price=("retail_price", "median"),
).round(2)

,products,avg_price,median_price
department,,,
Men,13131,63.21,44.00
Women,15989,55.94,37.46


In [7]:
# Categories: number of products and average price, most expensive first

(products
 .groupby(["department", "category"], observed=True)
 .agg(products=("id", "count"), avg_price=("retail_price", "mean"))
 .round(2)
 .sort_values("avg_price", ascending=False))

products  avg_price
department category                                          
Men        Outerwear & Coats                   828     149.27
Women      Outerwear & Coats                   592     141.47
Men        Suits & Sport Coats                 739     126.56
Women      Suits                               188     116.16
Men        Jeans                              1117     101.16
Women      Jeans                               882      93.66
           Blazers & Jackets                   561      92.60
           Clothing Sets                        37      84.84
           Dresses                             955      84.20
Men        Sweaters                            999      77.59
Women      Sweaters                            738      72.24
           Swim                                892      65.66
Men        Pants                              1041      59.81
           Fashion Hoodies & Sweatshirts       971      58.32
Women      Pants & Capris                      613      54.71
           Active                              679      53.93
           Skirts                              367      52.33
           Maternity                           898      50.80
Men        Sleep & Lounge                      941      50.16
           Shorts                              939      50.14
           Swim                                906      50.09
Women      Fashion Hoodies & Sweatshirts       895      49.14
           Sleep & Lounge                      830      48.15
Men        Active                              753      47.63
Women      Jumpsuits & Rompers                 162      45.42
           Accessories                         739      43.55
Men        Tops & Tees                        1084      42.25
           Accessories                         820      41.68
Women      Shorts                              826      40.80
           Tops & Tees                         784      40.08
           Plus                                758      38.43
           Intimates                          2363      33.71
Men        Underwear                          1088      27.16
Women      Leggings                            564      27.15
Men        Socks                               905      20.42
Women      Socks & Hosiery                     666      16.76

In [8]:
# Brands – how many, and which have the most products

print(f"Liczba marek: {products['brand'].nunique():,}")

products["brand"].value_counts().head(10)

Liczba marek: 2,756


brand
Allegra K         1034
Calvin Klein       497
Carhartt           388
Hanes              308
Volcom             292
Nautica            263
Levi's             259
Quiksilver         257
Tommy Hilfiger     251
Columbia           236
Name: count, dtype: int64

## 2. Marża katalogowa

* Marża % = (cena − koszt) / cena.
* Czy marża różni się między kategoriami? Jeśli min, średnia i max są wszędzie takie same,
  koszt został wygenerowany jako losowy % ceny – wtedy marża nie jest argumentem biznesowym.

In [9]:
# Margin % for every product

products["margin_pct"] = (products["retail_price"] - products["cost"]) / products["retail_price"] * 100

products["margin_pct"].describe().round(1)

count    29120.0
mean        51.1
std          7.2
min         33.0
25%         45.5
50%         51.2
75%         56.8
max         66.9
Name: margin_pct, dtype: float64

In [10]:
# Margin by category – mean, min and max

(products
 .groupby("category", observed=True)["margin_pct"]
 .agg(["mean", "min", "max"])
 .round(1)
 .sort_values("mean", ascending=False))

,mean,min,max
category,,,
Blazers & Jackets,62.0,57.0,66.9
Skirts,60.0,55.0,64.9
Socks & Hosiery,59.9,55.0,64.9
Suits & Sport Coats,59.9,55.0,64.9
Accessories,59.9,55.0,64.9
Active,58.0,53.0,62.9
Maternity,55.8,51.0,60.9
Outerwear & Coats,55.5,50.0,60.9
Dresses,54.9,50.0,59.9


## 3. Sprzedaż: przychód i zysk wg kategorii

Budujemy tabelę `sold`: **jeden wiersz = jedna sprzedana pozycja** (tylko Complete + Shipped),
z kategorią, działem, kosztem i płcią klienta.

In [11]:
# Sold items: order_items + order status/gender + product attributes

sold = (order_items
        .merge(orders, on="order_id", how="inner")
        .loc[lambda x: x["status"].isin(REVENUE_STATUSES)]
        .merge(products.drop(columns="margin_pct"), left_on="product_id", right_on="id",
               how="left", validate="many_to_one")
        .drop(columns="id")
        .assign(gross_profit=lambda x: x["sale_price"] - x["cost"]))

# Sanity checks: every item found its product, revenue comparable with A2
print(f"Sprzedane pozycje: {len(sold):,}, bez produktu: {sold['category'].isna().sum()}")
print(f"Przychód: {sold['sale_price'].sum():,.0f}")

sold.head()

Sprzedane pozycje: 99,808, bez produktu: 0
Przychód: 5,952,870


,order_id,product_id,sale_price,status,gender,brand,category,department,cost,retail_price,gross_profit
0,2526,14235,0.02,Complete,F,marshal,Accessories,Women,0.00830,0.02,0.01170
1,348,14235,0.02,Shipped,F,marshal,Accessories,Women,0.00830,0.02,0.01170
2,35767,14235,0.02,Shipped,F,marshal,Accessories,Women,0.00830,0.02,0.01170
3,86484,14159,0.49,Complete,F,Made in USA,Accessories,Women,0.17738,0.49,0.31262
4,21455,14159,0.49,Shipped,F,Made in USA,Accessories,Women,0.17738,0.49,0.31262


In [12]:
# Discounts – is sale_price ever different from the catalogue price?

(sold["sale_price"] - sold["retail_price"]).abs().gt(0.005).mean() * 100

np.float64(0.0)

In [13]:
# Category performance with Pareto: cumulative share of revenue

categories = (sold
              .groupby("category", observed=True)
              .agg(items=("sale_price", "count"),
                   revenue=("sale_price", "sum"),
                   gross_profit=("gross_profit", "sum"))
              .sort_values("revenue", ascending=False)
              .assign(margin_pct=lambda x: x["gross_profit"] / x["revenue"] * 100,
                      revenue_share_pct=lambda x: x["revenue"] / x["revenue"].sum() * 100,
                      cumulative_pct=lambda x: x["revenue_share_pct"].cumsum())
              .round(1))

categories

,items,revenue,gross_profit,margin_pct,revenue_share_pct,cumulative_pct
category,,,,,,
Outerwear & Coats,4897,724107.9,402171.1,55.5,12.2,12.2
Jeans,7102,698691.0,324795.7,46.5,11.7,23.9
Sweaters,6121,461907.3,239255.0,51.8,7.8,31.7
Suits & Sport Coats,2822,361641.9,216441.9,59.8,6.1,37.7
Fashion Hoodies & Sweatshirts,6553,357087.2,171502.5,48.0,6.0,43.7
Swim,6162,350283.0,172584.5,49.3,5.9,49.6
Sleep & Lounge,6198,303623.2,157589.7,51.9,5.1,54.7
Shorts,6149,284641.0,142170.4,49.9,4.8,59.5
Tops & Tees,6585,269313.6,118487.6,44.0,4.5,64.0


In [14]:
# Top 10 brands by revenue and their share of total revenue

(sold
 .groupby("brand", observed=True)
 .agg(items=("sale_price", "count"), revenue=("sale_price", "sum"))
 .sort_values("revenue", ascending=False)
 .assign(revenue_share_pct=lambda x: x["revenue"] / x["revenue"].sum() * 100)
 .head(10)
 .round(1))

,items,revenue,revenue_share_pct
brand,,,
Calvin Klein,1755,111865.9,1.9
True Religion,515,105437.8,1.8
Diesel,755,100009.2,1.7
Carhartt,1439,98028.4,1.6
7 For All Mankind,615,96731.5,1.6
Tommy Hilfiger,869,63428.2,1.1
Volcom,1029,61309.4,1.0
Oakley,495,56133.2,0.9
Columbia,801,55779.4,0.9


## 4. Hipoteza z A2: czy mężczyźni kupują inaczej, czy po prostu drożej?

AOV rozkładamy na dwa czynniki: **AOV = liczba pozycji w zamówieniu × średnia cena pozycji**.

* Jeśli różnica jest w liczbie pozycji → różne zachowanie (mężczyźni kupują więcej rzeczy naraz).
* Jeśli w cenie pozycji → sprawdzamy, czy mężczyźni kupują produkty z działu Men
  i czy te same kategorie są w dziale Men droższe.

In [15]:
# AOV decomposition by customer gender

(sold
 .groupby("gender", observed=True)
 .agg(orders=("order_id", "nunique"),
      items=("sale_price", "count"),
      revenue=("sale_price", "sum"))
 .assign(items_per_order=lambda x: x["items"] / x["orders"],
         avg_item_price=lambda x: x["revenue"] / x["items"],
         aov=lambda x: x["revenue"] / x["orders"])
 .round(2))

,orders,items,revenue,items_per_order,avg_item_price,aov
gender,,,,,,
F,34225,49681,2781425.05,1.45,55.99,81.27
M,34442,50127,3171444.61,1.46,63.27,92.08


In [16]:
# Which department do customers buy from? Row = customer gender, each row sums to 100%

pd.crosstab(sold["gender"], sold["department"], normalize="index").mul(100).round(1)

department,Men,Women
gender,,
F,0.0,100.0
M,100.0,0.0


In [17]:
# Same category, different department – is the Men version more expensive?
# Only categories that exist in both departments

price_by_dept = (sold
                 .pivot_table(index="category", columns="department", values="sale_price",
                              aggfunc="mean", observed=True)
                 .dropna()
                 .assign(men_vs_women_pct=lambda x: (x["Men"] / x["Women"] - 1) * 100)
                 .round(1)
                 .sort_values("men_vs_women_pct", ascending=False))

price_by_dept

department,Men,Women,men_vs_women_pct
category,,,
Shorts,50.2,40.8,23.0
Fashion Hoodies & Sweatshirts,58.9,48.9,20.4
Jeans,101.6,93.2,9.1
Sweaters,77.8,71.6,8.7
Tops & Tees,41.6,39.8,4.5
Sleep & Lounge,49.6,48.2,2.9
Outerwear & Coats,147.8,147.9,-0.1
Accessories,42.5,44.1,-3.6
Active,47.1,52.7,-10.6


In [18]:
# Category mix: share of each gender's revenue by category (top 10 for men)

mix = (sold
       .groupby(["gender", "category"], observed=True)["sale_price"].sum()
       .groupby(level="gender", group_keys=False)
       .apply(lambda s: s / s.sum() * 100)
       .unstack("gender")
       .round(1))

mix.sort_values("M", ascending=False).head(10)

gender,F,M
category,,
Outerwear & Coats,9.6,14.4
Jeans,9.1,14.1
Suits & Sport Coats,NaN,11.4
Sweaters,6.0,9.3
Pants,NaN,7.5
Fashion Hoodies & Sweatshirts,5.1,6.8
Sleep & Lounge,4.4,5.7
Shorts,3.8,5.7
Tops & Tees,3.5,5.4


## Wnioski

1. **Różnica AOV między płciami to efekt asortymentu, nie zachowania.** Pozycji w zamówieniu jest tyle samo
   (1,45 vs 1,46), różni się średnia cena pozycji (M 63,27 vs F 55,99). Kobiety kupują w 100% z działu Women,
   mężczyźni w 100% z Men – płeć i dział są nierozdzielne. Wyższy AOV mężczyzn wynika głównie ze struktury
   koszyka (garnitury 11,4% przychodu, kurtki i jeansy po ~14%).
2. **Marża różni się systematycznie między kategoriami** (każda ma własny przedział ~10 pp.):
   od ~60% (Blazers & Jackets, Accessories, Suits & Sport Coats, Skirts) do ~40% (Socks, Suits, Clothing Sets).
   Ranking po przychodzie ≠ ranking po zysku (Jeans: 2. miejsce w przychodzie, marża 46,5%)
   → budżet reklamowy warto optymalizować pod zysk brutto, nie przychód.
3. **Sprzedaż jest rozproszona:** 13 z 26 kategorii daje 80% przychodu; 2 756 marek, największa ma 1,9%,
   top 10 – 13,4%. Brak rabatów (sale_price = retail_price).